In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from collections import defaultdict
import warnings
import os
warnings.simplefilter(action="ignore", category=pd.errors.SettingWithCopyWarning)

def build_dp_table(mrc_dict, max_total_slabs, trace_names, access_freqs, pretty_print=False):
    """
    Builds the DP table and allocation table for the given trace names and maximum total slabs.

    Parameters:
    mrc_dict (dict): A nested dictionary that maps trace_name to their miss ratio at different slab_cnt.
    max_total_slabs (int): The maximum number of slabs to consider.
    trace_names (list): The trace names that we are interested in.
    access_freqs (list): The access frequencies for the trace names.
    pretty_print (bool): If True, pretty print the DP table and allocation table.

    Returns:
    tuple: (dp, allocation) where:
        - dp: The DP table storing the minimum weighted miss ratio for each trace and slab count.
        - allocation: The allocation table storing the number of slabs allocated to each trace.
    """
    # Number of traces
    n = len(trace_names)
    
    # Initialize the DP table
    dp = [[float('inf')] * (max_total_slabs + 1) for _ in range(n + 1)]
    dp[0][0] = 0  # Base case: 0 slabs for 0 traces has a miss ratio of 0
    
    # Initialize the allocation table
    allocation = [[0] * (max_total_slabs + 1) for _ in range(n + 1)]
    
    # Fill the DP table
    for i in range(1, n + 1):
        trace_name = trace_names[i - 1]
        access_freq = access_freqs[i - 1]
        for j in range(max_total_slabs + 1):
            for k in range(j + 1):
                miss_ratio = mrc_dict[trace_name].get(k, 1)
                miss_count = miss_ratio * access_freq
                if dp[i - 1][j - k] + miss_count < dp[i][j]:
                    dp[i][j] = dp[i - 1][j - k] + miss_count
                    allocation[i][j] = k
    
    # Pretty print the DP table and allocation table if requested
    if pretty_print:
        print("DP Table:")
        for row in dp:
            print(', '.join([f'{x:.4f}' for x in row]))
        print("\nAllocation Table:")
        for row in allocation:
            print(', '.join([f'{x:3d}' for x in row]))
    
    return dp, allocation


def backtrack_allocation(dp, allocation, trace_names, total_slabs, access_freqs):
    """
    Performs backtracking on the precomputed DP table to determine the optimal allocation for a given total_slabs.

    Parameters:
    dp (list): The DP table built by `build_dp_table`.
    allocation (list): The allocation table built by `build_dp_table`.
    trace_names (list): The trace names that we are interested in.
    total_slabs (int): The total number of slabs to allocate.
    access_freqs (list): The access frequencies for the trace names.

    Returns:
    tuple: (result, normalized_miss_ratio) where:
        - result: A dictionary with the optimal allocation of slabs for each trace name.
        - normalized_miss_ratio: The minimized weighted miss ratio normalized by the total access frequency.
    """
    # Number of traces
    n = len(trace_names)
    
    # Backtrack to find the optimal allocation
    result = {}
    j = total_slabs
    for i in range(n, 0, -1):
        trace_name = trace_names[i - 1]
        result[trace_name] = allocation[i][j]
        j -= allocation[i][j]
    
    # Normalized miss ratio
    normalized_miss_ratio = dp[n][total_slabs] / sum(access_freqs)
    
    return result, normalized_miss_ratio



def compute_optimal_allocations(mrc_dict, mrc_delta_dict, wss_slabs_dict, max_total_slabs, trace_names, access_freqs):
    """
    Compute the optimal slab allocations and miss ratios for each total_slab from 1 to max_total_slabs.

    Parameters:
    mrc_dict (dict): A nested dictionary that maps trace_name to their miss ratio at different slab_cnt.
    max_total_slabs (int): The maximum number of slabs to consider.
    trace_names (list): The trace names that we are interested in.
    access_freqs (list): The access frequencies for the trace names.

    Returns:
    pd.DataFrame: A DataFrame where each row corresponds to a total_slab and contains:
        - Columns for each trace_name (number of slabs allocated to the trace).
        - 'total_miss_ratio': The normalized miss ratio for the given total_slab.
        - 'total_slab_cnt': The total number of slabs.
    """

    dp, allocation = build_dp_table(mrc_dict, max_total_slabs, trace_names, access_freqs)


    results = []
    for total_slab in range(1, max_total_slabs + 1):
        alloc, miss_ratio = backtrack_allocation(dp, allocation, trace_names, total_slab, access_freqs)
        # no more increase after it saturates
        row = {trace_name: min(alloc[trace_name], wss_slabs_dict[trace_name]) for trace_name in trace_names}
        row['total_miss_ratio'] = miss_ratio
        row['total_slab_cnt'] = total_slab
        for trace_name in trace_names:
            row[f"{trace_name}_miss_ratio"] = mrc_dict[trace_name][alloc[trace_name]]
            row[f"{trace_name}_miss_ratio_delta"] = mrc_delta_dict[trace_name][alloc[trace_name]]
        results.append(row)

    results_df = pd.DataFrame(results)
    return results_df

In [2]:
import heapq
import pandas as pd

def greedy_allocation_with_snapshots(mrc_dict, mrc_delta_dict, wss_slabs_dict, max_total_slabs, trace_names, access_freqs):
    """
    Greedy approach to allocate slabs based on utility, with tracking of allocation order and snapshots.

    Parameters:
    mrc_dict (dict): A nested dictionary that maps trace_name to their miss ratio at different slab counts.
    mrc_delta_dict (dict): A nested dictionary that maps trace_name to the reduction in miss ratio (delta) for each additional slab.
    max_total_slabs (int): The maximum number of slabs to allocate.
    trace_names (list): The trace names (class names) to allocate slabs to.
    access_freqs (list): The access frequencies for each trace.

    Returns:
    tuple: (allocation, normalized_miss_ratio, allocation_order, snapshots_df) where:
        - allocation: A dictionary mapping each trace_name to the number of slabs allocated.
        - normalized_miss_ratio: The normalized miss ratio after all slabs are allocated.
        - allocation_order: A list tracking the order in which slabs were allocated to traces.
        - snapshots_df: A DataFrame where each row corresponds to a snapshot of the allocation at a given total_slab.
    """

    allocation = {trace_name: 0 for trace_name in trace_names}
    allocation_order = []  
    snapshots = []  

    max_heap = []
    for i, trace_name in enumerate(trace_names):
        utility = mrc_delta_dict[trace_name][1] * access_freqs[i]
        heapq.heappush(max_heap, (-utility, False, i, trace_name))


    for total_slab in range(1, max_total_slabs + 1):
        if not max_heap:
            break  

        neg_utility, index, saturated, trace_name = heapq.heappop(max_heap)
        current_slabs = allocation[trace_name]
        allocation[trace_name] += 1  
        allocation_order.append(trace_name)  

        next_slabs = current_slabs + 1
        if next_slabs + 1 in mrc_delta_dict[trace_name]:  
            next_utility = mrc_delta_dict[trace_name][next_slabs + 1] * access_freqs[index]
            # Push (-utility, index, trace_name) to the heap to maintain tie-breaking
            heapq.heappush(max_heap, (-next_utility, index, next_slabs >= wss_slabs_dict[trace_name], trace_name))
        # no more increase after it saturates
        snapshot = {trace_name: min(allocation[trace_name], wss_slabs_dict[trace_name]) for trace_name in trace_names}
        snapshot['total_slab_cnt'] = total_slab
        snapshot['total_miss_ratio'] = sum(
            mrc_dict[trace_name][allocation[trace_name]] * access_freqs[i]
            for i, trace_name in enumerate(trace_names)
        ) / sum(access_freqs)
        for trace_name in trace_names:
            snapshot[f"{trace_name}_miss_ratio"] = mrc_dict[trace_name][allocation[trace_name]]
            snapshot[f"{trace_name}_miss_ratio_delta"] = (
                mrc_delta_dict[trace_name][allocation[trace_name]]
                if allocation[trace_name] in mrc_delta_dict[trace_name]
                else 0
            )
        snapshots.append(snapshot)

    # Calculate the normalized miss ratio
    total_miss_ratio = 0
    total_access_freq = sum(access_freqs)
    for i, trace_name in enumerate(trace_names):
        slabs_allocated = allocation[trace_name]
        miss_ratio = mrc_dict[trace_name][slabs_allocated]
        total_miss_ratio += miss_ratio * access_freqs[i]

    normalized_miss_ratio = total_miss_ratio / total_access_freq

    # Convert snapshots to a DataFrame
    snapshots_df = pd.DataFrame(snapshots)

    return allocation, normalized_miss_ratio, allocation_order, snapshots_df

In [3]:
import re
def process_chunked_subtraces(directory):
    
    chunk_dirs = []
    pattern = re.compile(r'^chunk_\d+$')  # Regex to match 'chunk_xxx' where xxx are digits

    for subdir in os.listdir(directory):
        subdir_path = os.path.join(directory, subdir)
        if os.path.isdir(subdir_path) and pattern.match(subdir):
            chunk_dirs.append(subdir)
    if not chunk_dirs:
        chunk_dirs = [directory]
    
    optimal_dp_miss_ratios = []
    optimal_dp_allocations = {}
    optimal_greedy_miss_ratios = []
    optimal_greedy_allocations = {}
    
    for chunk_dir in chunk_dirs:
        chunk_path = os.path.join(directory, chunk_dir)
        miss_ratios_path = os.path.join(chunk_path, "miss_ratios.csv")
        subtrace_stat_path = os.path.join(chunk_path, "subtrace_stat.csv")
        
        subtrace_miss_ratio_df = pd.read_csv(miss_ratios_path)
        subtrace_stat_df = pd.read_csv(subtrace_stat_path) 

       
        subtrace_miss_ratio_df['class_size'] = subtrace_miss_ratio_df['subtrace_name'].map(lambda x: int(x.split('.')[0].split('_')[-1]))
        subtrace_stat_df['class_size'] = subtrace_stat_df['subtrace_name'].map(lambda x: int(x.split('.')[0].split('_')[-1]))
        subtrace_stat_df['wss_slabs'] = np.ceil((subtrace_stat_df['distinct_object_count'] * subtrace_stat_df['class_size']) / (4 * 1024 * 1024)).astype(int)
        
        
        records = subtrace_miss_ratio_df.to_dict(orient='records')
        mrc_dict = defaultdict(dict)
        mrc_delta_dict = defaultdict(dict)

        for record in records:
            mrc_dict[record['class_size']][record['slab_cnt']] = record['miss_ratio']
            mrc_delta_dict[record['class_size']][record['slab_cnt']] = record['miss_ratio_delta']
            

        for class_size in mrc_dict:
            mrc_dict[class_size][0] = 1
            mrc_delta_dict[class_size][0] = float('inf')

        wss_slabs_dict = {}
        for record in subtrace_stat_df.to_dict(orient='records'):
            wss_slabs_dict[record['class_size']] = record['wss_slabs']

        class_sizes = sorted(mrc_dict.keys())
        
        access_freqs = {
            r['class_size']: r['record_count']
            for r in subtrace_stat_df.to_dict(orient='records')
        }

        slab_upper_limit = 1024
        optim_allocs_df = compute_optimal_allocations(mrc_dict, mrc_delta_dict, wss_slabs_dict, slab_upper_limit, list(mrc_dict.keys()), [access_freqs[k] for k in mrc_dict.keys()])
        _, _, _, greedy_snapshots_df = greedy_allocation_with_snapshots(
            mrc_dict, mrc_delta_dict, wss_slabs_dict, slab_upper_limit, sorted(list(mrc_dict.keys()), reverse=True), [access_freqs[k] for k in sorted(list(mrc_dict.keys()), reverse=True)]
        )

        total_records_cnt = sum(access_freqs.values())
        optimal_dp_miss_ratios.append((total_records_cnt, {r['total_slab_cnt']: r['total_miss_ratio'] for r in optim_allocs_df.to_dict(orient='records')}))
        for r in optim_allocs_df.to_dict(orient='records'):
            optimal_dp_allocations[(chunk_dir, r['total_slab_cnt'])] = {k: v for k, v in r.items() if k in class_sizes}
        optimal_greedy_miss_ratios.append((total_records_cnt, {r['total_slab_cnt']: r['total_miss_ratio'] for r in greedy_snapshots_df.to_dict(orient='records')}))
        for r in greedy_snapshots_df.to_dict(orient='records'):
            optimal_greedy_allocations[(chunk_dir, r['total_slab_cnt'])] = {k: v for k, v in r.items() if k in class_sizes}
    def compute_weighted_averages(miss_ratios):
        weighted_averages = defaultdict(float)
        total_records_per_slab = defaultdict(int)

        for total_records_cnt, miss_ratio_dict in miss_ratios:
            for total_slab, miss_ratio in miss_ratio_dict.items():
                weighted_averages[total_slab] += total_records_cnt * miss_ratio
                total_records_per_slab[total_slab] += total_records_cnt

        for total_slab in weighted_averages:
            weighted_averages[total_slab] /= total_records_per_slab[total_slab]

        return dict(weighted_averages)

    weighted_avg_dp = compute_weighted_averages(optimal_dp_miss_ratios)
    weighted_avg_greedy = compute_weighted_averages(optimal_greedy_miss_ratios)

    return weighted_avg_dp, weighted_avg_greedy, optimal_dp_allocations, optimal_greedy_allocations
    

In [55]:

trace_name = 'synth_thesis_static_104'


simulation_path = "report.csv"
simulation_df = pd.read_csv(simulation_path)
simulation_df = simulation_df[simulation_df['trace_name'] == trace_name]
simulation_df['slab_cnt'] = (simulation_df['cacheSizeMB'] - 4) // 4


base_dir = f"/mydata/hongshu/traces/thesis/subtraces/{trace_name}/chunk_0"
miss_ratios_path = os.path.join(base_dir, "miss_ratios.csv")
subtrace_stat_path = os.path.join(base_dir, "subtrace_stat.csv")

subtrace_miss_ratio_df = pd.read_csv(miss_ratios_path) if os.path.exists(miss_ratios_path) else None
subtrace_stat_df = pd.read_csv(subtrace_stat_path) if os.path.exists(subtrace_stat_path) else None


subtrace_miss_ratio_df['class_size'] = subtrace_miss_ratio_df['subtrace_name'].map(lambda x: int(x.split('.')[0].split('_')[-1]))
subtrace_stat_df['class_size'] = subtrace_stat_df['subtrace_name'].map(lambda x: int(x.split('.')[0].split('_')[-1]))
subtrace_stat_df['wss_slabs'] = np.ceil((subtrace_stat_df['distinct_object_count'] * subtrace_stat_df['class_size']) / (4 * 1024 * 1024)).astype(int)
subtrace_stat_df['wss'] = (subtrace_stat_df['distinct_object_count'] * subtrace_stat_df['class_size'])


optimal_lookup_dict, greedy_optimal_lookup_dict, optimal_dp_allocations, optimal_greedy_allocations\
    = process_chunked_subtraces(f"/mydata/hongshu/traces/thesis/subtraces/{trace_name}/")

optimal_allocs = []
for slab, mr in optimal_lookup_dict.items():
    alloc = optimal_dp_allocations.get(('chunk_0', slab), {})
    alloc['total_slab_cnt'] = slab
    alloc['total_miss_ratio'] = mr
    optimal_allocs.append(alloc)
optimal_allocs_df = pd.DataFrame(optimal_allocs)

greedy_allocs = []
for slab, mr in greedy_optimal_lookup_dict.items():
    alloc = optimal_greedy_allocations.get(('chunk_0', slab), {})
    alloc['total_slab_cnt'] = slab
    alloc['total_miss_ratio'] = mr
    greedy_allocs.append(alloc)
greedy_allocs_df = pd.DataFrame(greedy_allocs)



In [56]:
optimal_allocs_df[optimal_allocs_df['total_slab_cnt'] == 32]

,2048,4096,total_slab_cnt,total_miss_ratio
31,5,27,32,0.64637


In [57]:
import ast

def parse_acStats(acStats_str):
    try:
        parsed_list = ast.literal_eval(acStats_str)
        return {entry['allocSize']: entry['totalSlabs'] for entry in parsed_list}
    except Exception as e:
        print(f"Error parsing: {acStats_str}, Error: {e}")
        return {}


analysis_df = simulation_df[simulation_df['trace_name'] == trace_name]
analysis_df['class_slabs'] = analysis_df['_acStats'].apply(parse_acStats)


In [58]:
analysis_df.sort_values(by='slab_cnt', inplace=True)
analysis_df[['slab_cnt', 'class_slabs', '_missRatio']]

,slab_cnt,class_slabs,_missRatio
11,32,"{2048: 6, 4096: 26}",0.667924
0,64,"{2048: 11, 4096: 53}",0.620433
7,128,"{2048: 21, 4096: 107}",0.549840
21,256,"{2048: 39, 4096: 217}",0.439060
4,512,"{2048: 79, 4096: 433}",0.255981


In [24]:
def compare_with_optimal(analysis_df, optimal_allocs_df, slab_cnt):
    row = analysis_df[analysis_df['slab_cnt'] == slab_cnt]
    if row.empty:
        raise ValueError(f"No row found in analysis_df for slab_cnt={slab_cnt}")
    row = row.iloc[0]
    class_slabs = row['class_slabs']  # dict like {2048: 21, 4096: 43}
    miss_ratio = row['_missRatio']

    # Find the corresponding row in optimal_allocs_df
    opt_row = optimal_allocs_df[optimal_allocs_df['total_slab_cnt'] == slab_cnt]
    if opt_row.empty:
        raise ValueError(f"No row found in optimal_allocs_df for total_slab_cnt={slab_cnt}")
    opt_row = opt_row.iloc[0]
    optimal_miss_ratio = opt_row['total_miss_ratio']

    # Compute miss ratio diff
    miss_ratio_diff = miss_ratio - optimal_miss_ratio

    # Compute allocation distance (Manhattan distance / 2)
    alloc_distance = 0
    for k in class_slabs:
        alloc_distance += abs(class_slabs.get(k, 0) - opt_row.get(k, 0))
    alloc_distance /= 2

    return {
        'miss_ratio': float(miss_ratio),
        'optimal_miss_ratio': float(optimal_miss_ratio),
        'miss_ratio_diff': float(miss_ratio_diff),
        'miss_ratio_diff_pct': float(miss_ratio_diff / optimal_miss_ratio) if optimal_miss_ratio != 0 else float('inf'),
        'alloc_distance': int(alloc_distance),
        'class_slabs': class_slabs,
        'optimal_alloc': {k: float(opt_row.get(k, 0)) for k in class_slabs}
    }

In [26]:
compare_with_optimal(analysis_df, optimal_allocs_df, 64)

{'miss_ratio': 0.62043275,
 'optimal_miss_ratio': 0.5948908,
 'miss_ratio_diff': 0.025541949999999924,
 'miss_ratio_diff_pct': 0.04293552699083583,
 'alloc_distance': 26,
 'class_slabs': {2048: 11, 4096: 53},
 'optimal_alloc': {2048: 37.0, 4096: 27.0}}